# Домашняя работа 29: работа с большой языковой моделью Saiga

## Цель работы

В этой домашней работе работа с большими языковыми моделями и Hugging Face.

В работе используется русскоязычная модель Saiga на базе Mistral 7B.

## Что будет сделано

1. Создана учетная запись Hugging Face.
2. Получен Hugging Face token.
3. Загружена модель Saiga.
4. Сгенерированы 3 шутки с помощью Saiga.
5. Использована квантованная версия модели Saiga Mistral 7B GGUF.
6. Для дополнительной оценки качества используется датасет ru_turbo_saiga.

## Почему используется GGUF

GGUF - это формат квантованных моделей, который позволяет запускать большие языковые модели с меньшими требованиями к памяти. Это удобно для Colab, потому что полная модель 7B может требовать много видеопамяти.

In [ ]:
# проверка среды
import sys
import platform

print("Python:", sys.version)
print("Платформа:", platform.platform())

try:
    import torch
    print("PyTorch установлен:", torch.__version__)
    print("CUDA доступна:", torch.cuda.is_available())

    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
    else:
        print("GPU не обнаружен. Модель всё равно можно запускать через CPU, но будет медленнее.")
except Exception as e:
    print("PyTorch не найден или возникла ошибка:", e)

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Платформа: Linux-6.6.122+-x86_64-with-glibc2.35
PyTorch установлен: 2.10.0+cu128
CUDA доступна: True
GPU: Tesla T4


In [ ]:
# Установка библиотек для работы с Saiga GGUF и датасетом

!pip -q install --upgrade pip

# llama-cpp-python с поддержкой CUDA.
# Если эта строка установится успешно, модель сможет использовать GPU.
!pip -q install llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124

# Библиотеки Hugging Face и оценки качества
!pip -q install huggingface_hub datasets sentence-transformers pandas tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 83.2 MB/s eta 0:00:00


In [ ]:
# проверки
import llama_cpp
import datasets
import sentence_transformers
import pandas as pd

print("llama_cpp установлен:", llama_cpp.__version__)
print("datasets установлен")
print("sentence_transformers установлен")
print("pandas установлен:", pd.__version__)

llama_cpp установлен: 0.3.23
datasets установлен
sentence_transformers установлен
pandas установлен: 2.2.2


In [ ]:
# Авторизация Hugging
# Пункт 5. Авторизация в Hugging Face

import os
import getpass
from huggingface_hub import login, whoami

hf_token = getpass.getpass("Введите Hugging Face token, который начинается с hf_: ").strip()

if not hf_token.startswith("hf_"):
    raise ValueError("Вы вставили не Hugging Face token. Токен должен начинаться с hf_.")

login(token=hf_token, add_to_git_credential=False)

os.environ["HF_TOKEN"] = hf_token

user_info = whoami(token=hf_token)

print("Авторизация Hugging Face выполнена успешно.")
print("Аккаунт:", user_info.get("name"))


Введите Hugging Face token, который начинается с hf_: ··········
Авторизация Hugging Face выполнена успешно.
Аккаунт: DisaUFA


In [ ]:
# Проверка доступа к модели и датасету

from huggingface_hub import list_repo_files

MODEL_REPO = "IlyaGusev/saiga_mistral_7b_gguf"
DATASET_REPO = "IlyaGusev/ru_turbo_saiga"

print("Проверяем файлы модели Saiga GGUF...")
model_files = list_repo_files(
    repo_id=MODEL_REPO,
    repo_type="model",
    token=hf_token
)

print("Файлы модели:")
for f in model_files:
    print("-", f)

print("\nПроверяем файлы датасета ru_turbo_saiga...")
dataset_files = list_repo_files(
    repo_id=DATASET_REPO,
    repo_type="dataset",
    token=hf_token
)

print("Файлы датасета:")
for f in dataset_files:
    print("-", f)

print("\nПроверка завершена успешно.")

Проверяем файлы модели Saiga GGUF...
Файлы модели:
- .gitattributes
- README.md
- model-q2_K.gguf
- model-q4_K.gguf
- model-q8_0.gguf

Проверяем файлы датасета ru_turbo_saiga...
Файлы датасета:
- .gitattributes
- README.md
- ru_turbo_saiga.jsonl.zst
- ru_turbo_saiga.py

Проверка завершена успешно.


## Загрузка квантованной модели Saiga Mistral 7B GGUF

Для выполнения задания используется квантованная версия модели Saiga Mistral 7B в формате GGUF.

Я выбираю файл `model-q4_K.gguf`, потому что это компромисс между качеством ответа и требованиями к памяти. Формат GGUF позволяет запускать большую языковую модель с меньшими требованиями к ресурсам.

In [ ]:
# Загрузка квантованной модели Saiga Mistral 7B GGUF

import gc
import torch
from llama_cpp import Llama

MODEL_REPO = "IlyaGusev/saiga_mistral_7b_gguf"
MODEL_FILE = "model-q4_K.gguf"

# Очищаем память перед загрузкой модели
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("GPU доступен:", torch.cuda.get_device_name(0))
else:
    print("GPU не найден. Модель будет работать на CPU.")

print("Начинаем загрузку модели:", MODEL_REPO)
print("Файл модели:", MODEL_FILE)

llm = Llama.from_pretrained(
    repo_id=MODEL_REPO,
    filename=MODEL_FILE,
    n_ctx=2048,
    n_gpu_layers=-1,
    n_threads=4,
    verbose=True
)

print("Модель Saiga Mistral 7B GGUF успешно загружена.")

GPU доступен: Tesla T4
Начинаем загрузку модели: IlyaGusev/saiga_mistral_7b_gguf
Файл модели: model-q4_K.gguf


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


./model-q4_K.gguf:   0%|          | 0.00/4.37G [00:00<?, ?B/s]

ggml_cuda_init: found 1 CUDA devices (Total VRAM: 14912 MiB):
  Device 0: Tesla T4, compute capability 7.5, VMM: yes, VRAM: 14912 MiB
llama_model_loader: loaded meta data with 21 key-value pairs and 291 tensors from /root/.cache/huggingface/hub/models--IlyaGusev--saiga_mistral_7b_gguf/snapshots/0ebc67baa384506f0530b7cba8b43a10584a22bc/./model-q4_K.gguf (version GGUF V2)
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.name str              = models
llama_model_loader: - kv   2:                       llama.context_length u32              = 32768
llama_model_loader: - kv   3:                     llama.embedding_length u32              = 4096
llama_model_loader: - kv   4:                          llama.block_count u32              = 32
llama_model_loader: - kv   5:             

Модель Saiga Mistral 7B GGUF успешно загружена.


CUDA : ARCHS = 700,750,800,860,890,900 | FORCE_MMQ = 1 | USE_GRAPHS = 1 | PEER_MAX_BATCH_SIZE = 128 | CPU : SSE3 = 1 | SSSE3 = 1 | AVX = 1 | AVX2 = 1 | F16C = 1 | FMA = 1 | BMI2 = 1 | LLAMAFILE = 1 | OPENMP = 1 | REPACK = 1 | 
Model metadata: {'tokenizer.ggml.padding_token_id': '0', 'tokenizer.ggml.unknown_token_id': '0', 'tokenizer.ggml.eos_token_id': '2', 'general.architecture': 'llama', 'llama.rope.freq_base': '10000.000000', 'llama.context_length': '32768', 'general.name': 'models', 'llama.embedding_length': '4096', 'llama.feed_forward_length': '14336', 'llama.attention.layer_norm_rms_epsilon': '0.000010', 'llama.rope.dimension_count': '128', 'tokenizer.ggml.bos_token_id': '1', 'llama.attention.head_count': '32', 'llama.block_count': '32', 'llama.attention.head_count_kv': '8', 'general.quantization_version': '2', 'tokenizer.ggml.model': 'llama', 'general.file_type': '15'}
Using fallback chat format: llama-2


## Генерация 3 шуток с помощью Saiga

На этом этапе я проверяю, что модель Saiga действительно умеет генерировать ответы на русском языке.  
Для выполнения задания на 3 балла нужно сгенерировать 3 шутки.

In [ ]:
# Генерация 3 шуток с помощью Saiga

def make_saiga_prompt(user_text):
    system_text = (
        "Ты русскоязычный ассистент. "
        "Отвечай на русском языке кратко, понятно и без лишних пояснений."
    )

    prompt = f"""<|im_start|>system
{system_text}<|im_end|>
<|im_start|>user
{user_text}<|im_end|>
<|im_start|>assistant
"""
    return prompt


def generate_saiga_answer(user_text, max_tokens=180, temperature=0.8):
    prompt = make_saiga_prompt(user_text)

    output = llm(
        prompt,
        max_tokens=max_tokens,
        temperature=temperature,
        top_p=0.9,
        repeat_penalty=1.1,
        stop=["<|im_end|>", "</s>"]
    )

    answer = output["choices"][0]["text"].strip()
    return answer


joke_prompts = [
    "Придумай короткую шутку про программиста.",
    "Придумай короткую шутку про нейросети и фриланс.",
    "Придумай короткую шутку про Hugging Face и Google Colab."
]

jokes = []

for i, prompt_text in enumerate(joke_prompts, start=1):
    print(f"\n--- Шутка {i} ---")
    print("Запрос:", prompt_text)

    answer = generate_saiga_answer(prompt_text)
    jokes.append(answer)

    print("Ответ Saiga:")
    print(answer)

Llama.generate: 1 prefix-match hit, remaining 73 prompt tokens to eval
ggml_cuda_graph_set_enabled: disabling CUDA graphs due to GPU architecture



--- Шутка 1 ---
Запрос: Придумай короткую шутку про программиста.


llama_perf_context_print:        load time =     659.87 ms
llama_perf_context_print: prompt eval time =     144.38 ms /    73 tokens (    1.98 ms per token,   505.62 tokens per second)
llama_perf_context_print:        eval time =    1327.30 ms /    35 runs   (   37.92 ms per token,    26.37 tokens per second)
llama_perf_context_print:       total time =    1502.47 ms /   108 tokens
llama_perf_context_print:    graphs reused =         34
Llama.generate: 63 prefix-match hit, remaining 17 prompt tokens to eval


Ответ Saiga:
Программист потерял свою закладную пароль, и решил узнать его у другого программиста.

--- Шутка 2 ---
Запрос: Придумай короткую шутку про нейросети и фриланс.


llama_perf_context_print:        load time =     659.87 ms
llama_perf_context_print: prompt eval time =      60.65 ms /    17 tokens (    3.57 ms per token,   280.29 tokens per second)
llama_perf_context_print:        eval time =    1349.08 ms /    35 runs   (   38.55 ms per token,    25.94 tokens per second)
llama_perf_context_print:       total time =    1442.43 ms /    52 tokens
llama_perf_context_print:    graphs reused =         34
Llama.generate: 63 prefix-match hit, remaining 14 prompt tokens to eval


Ответ Saiga:
Фрилансер, который работает с нейросетью, забыл о том, что нельзя делать это удаленно.

--- Шутка 3 ---
Запрос: Придумай короткую шутку про Hugging Face и Google Colab.


llama_perf_context_print:        load time =     659.87 ms
llama_perf_context_print: prompt eval time =      59.09 ms /    14 tokens (    4.22 ms per token,   236.91 tokens per second)
llama_perf_context_print:        eval time =    2436.12 ms /    62 runs   (   39.29 ms per token,    25.45 tokens per second)
llama_perf_context_print:       total time =    2541.06 ms /    76 tokens
llama_perf_context_print:    graphs reused =         61


Ответ Saiga:
Hugging Face: "Мы обнимаем лица, а не лживые маски."

Google Colab: "Они собирают мощные компьютеры, чтобы помогать нам вычислить тот самый Google Colab."


## Загрузка датасета ru_turbo_saiga

Для дополнительной оценки качества модели используется датасет `IlyaGusev/ru_turbo_saiga`.

Сначала я загружаю несколько примеров и изучаю структуру датасета: какие поля есть в записи, где находится вопрос пользователя и где находится эталонный ответ ассистента.

In [ ]:
# Загрузка датасета ru_turbo_saiga и просмотр структуры


from datasets import load_dataset
from itertools import islice
import json

DATASET_FILE = "hf://datasets/IlyaGusev/ru_turbo_saiga/ru_turbo_saiga.jsonl.zst"

print("Загружаем файл датасета:")
print(DATASET_FILE)

dataset_stream = load_dataset(
    "json",
    data_files=DATASET_FILE,
    split="train",
    streaming=True,
    token=hf_token
)

examples = list(islice(dataset_stream, 3))

print("Количество загруженных примеров для просмотра:", len(examples))

for i, example in enumerate(examples, start=1):
    print(f"\n--- Пример {i} ---")
    print("Ключи записи:", list(example.keys()))
    print(json.dumps(example, ensure_ascii=False, indent=2)[:3000])

Загружаем файл датасета:
hf://datasets/IlyaGusev/ru_turbo_saiga/ru_turbo_saiga.jsonl.zst
Количество загруженных примеров для просмотра: 3

--- Пример 1 ---
Ключи записи: ['seed', 'source', 'messages', 'model_name']
{
  "seed": "Как называется фильм в котором детей из детдомв набирают в группу для спец операций??",
  "source": "yandex_q",
  "messages": [
    {
      "role": "user",
      "content": "Мне нужно найти название фильма. В нем детей из детдомов набирают в группу для спец операций. Ты можешь мне помочь?"
    },
    {
      "role": "bot",
      "content": "Конечно. Навскидку я могу предложить несколько фильмов. Но, скорее всего, ты ищешь фильм под названием \"Сиротский батальон\" (англ. \"The Little Bastards\"). Эту картина, снятая в 2003 году, рассказывает о группе детей совершающих различные военные миссии, посланные на линию фронта."
    },
    {
      "role": "user",
      "content": "Да, спасибо. Извини, не было времени на поиск поисковиком. А есть ли другие фильмы на эту 

## Подготовка тестовых примеров для оценки

В датасете `ru_turbo_saiga` каждая запись содержит список сообщений `messages`.

Для оценки я беру первую пару:

- сообщение пользователя `role = user`;
- следующий за ним эталонный ответ ассистента `role = bot`.

Далее эти вопросы будут переданы модели Saiga, а ответы модели будут сравнены с эталонными ответами из датасета.

In [ ]:
# Подготовка тестовых пар: вопрос пользователя - эталонный ответ

from datasets import load_dataset
from itertools import islice
import pandas as pd

DATASET_FILE = "hf://datasets/IlyaGusev/ru_turbo_saiga/ru_turbo_saiga.jsonl.zst"

dataset_stream = load_dataset(
    "json",
    data_files=DATASET_FILE,
    split="train",
    streaming=True,
    token=hf_token
)


def extract_first_user_bot_pair(example):
    """
    Из одной записи датасета достаём первую пару:
    user -> bot.
    """

    messages = example.get("messages", [])

    for i in range(len(messages) - 1):
        current_message = messages[i]
        next_message = messages[i + 1]

        if current_message.get("role") == "user" and next_message.get("role") == "bot":
            return {
                "source": example.get("source", ""),
                "question": current_message.get("content", "").strip(),
                "reference_answer": next_message.get("content", "").strip()
            }

    return None


eval_examples = []

# Берём не больше 5 примеров, чтобы оценка не выполнялась слишком долго
for example in dataset_stream:
    pair = extract_first_user_bot_pair(example)

    if pair is None:
        continue

    if len(pair["question"]) < 20:
        continue

    if len(pair["reference_answer"]) < 20:
        continue

    eval_examples.append(pair)

    if len(eval_examples) >= 5:
        break


eval_df = pd.DataFrame(eval_examples)

print("Количество подготовленных тестовых примеров:", len(eval_df))
display(eval_df)

Количество подготовленных тестовых примеров: 5


,source,question,reference_answer
0,yandex_q,Мне нужно найти название фильма. В нем детей и...,Конечно. Навскидку я могу предложить несколько...
1,yandex_q,"Я слышал, что Петр I ввёл крепостное право в Р...","Да, Петр I действительно ввёл крепостное право..."
2,ru_stackoverflow,"Я хотел бы узнать больше о том, как использова...","Конечно, с удовольствием помогу вам! Счётчики ..."
3,yandex_q,Я хочу добавить 4 гб оперативной памяти на сво...,Для вашего ноутбука подходит оперативная памят...
4,ru_stackoverflow,"Я хочу настроить aiogram бота и не знаю, как п...","Конечно, я готов помочь вам с этим. Давайте на..."


## Генерация ответов Saiga на тестовых вопросах

На этом этапе я передаю модели Saiga вопросы из датасета `ru_turbo_saiga`.

Для каждого вопроса сохраняются:

- исходный вопрос пользователя;
- эталонный ответ из датасета;
- ответ, который сгенерировала модель Saiga.

Далее эти ответы будут использованы для оценки качества модели.

In [ ]:
# Генерация ответов Saiga на тестовых вопросах

from tqdm.auto import tqdm
import pandas as pd

model_answers = []

for index, row in tqdm(eval_df.iterrows(), total=len(eval_df)):
    question = row["question"]

    prompt_for_model = f"""
Ответь на вопрос пользователя на русском языке.
Ответ должен быть полезным, понятным и не слишком длинным.

Вопрос пользователя:
{question}
"""

    print(f"\n--- Пример {index + 1} ---")
    print("Вопрос:")
    print(question)

    answer = generate_saiga_answer(
        prompt_for_model,
        max_tokens=220,
        temperature=0.2
    )

    model_answers.append(answer)

    print("\nОтвет Saiga:")
    print(answer)


eval_results_df = eval_df.copy()
eval_results_df["model_answer"] = model_answers

print("\nГенерация ответов завершена.")
display(eval_results_df[["source", "question", "reference_answer", "model_answer"]])

  0%|          | 0/5 [00:00<?, ?it/s]

Llama.generate: 50 prefix-match hit, remaining 109 prompt tokens to eval
ggml_cuda_graph_set_enabled: disabling CUDA graphs due to GPU architecture



--- Пример 1 ---
Вопрос:
Мне нужно найти название фильма. В нем детей из детдомов набирают в группу для спец операций. Ты можешь мне помочь?


llama_perf_context_print:        load time =     659.87 ms
llama_perf_context_print: prompt eval time =     190.96 ms /   109 tokens (    1.75 ms per token,   570.81 tokens per second)
llama_perf_context_print:        eval time =    3435.27 ms /   105 runs   (   32.72 ms per token,    30.57 tokens per second)
llama_perf_context_print:       total time =    3726.25 ms /   214 tokens
llama_perf_context_print:    graphs reused =        103
Llama.generate: 106 prefix-match hit, remaining 44 prompt tokens to eval



Ответ Saiga:
Да, конечно, я могу помочь. Фильм с таким сюжетом называется "Операция «Спасти тьму»" (Operation: Endgame). Это американский боевик 2016 года, в котором актёры Джейсон Стейтемам и Наоми Уоттс играют роли солдат, которые должны спасти детей из детдомов.

--- Пример 2 ---
Вопрос:
Я слышал, что Петр I ввёл крепостное право в России после своей поездки в Европу. Это правда?


...

llama_perf_context_print:        load time =     659.87 ms
llama_perf_context_print: prompt eval time =      88.48 ms /    54 tokens (    1.64 ms per token,   610.33 tokens per second)
llama_perf_context_print:        eval time =    3516.50 ms /    96 runs   (   36.63 ms per token,    27.30 tokens per second)
llama_perf_context_print:       total time =    3677.03 ms /   150 tokens
llama_perf_context_print:    graphs reused =         94
Llama.generate: 109 prefix-match hit, remaining 43 prompt tokens to eval



Ответ Saiga:
Лучше всего подойдет модуль оперативной памяти DDR4 с частотой 2666 МГц или выше, который поддерживается вашим ноутбуком ASUS FX570UD-DM148T. Рекомендуется использовать два модуля по 2 ГБ каждый для общим объемом 4 ГБ.

--- Пример 5 ---
Вопрос:
Я хочу настроить aiogram бота и не знаю, как правильно построить архитектуру файлов для него. Можете помочь?


llama_perf_context_print:        load time =     659.87 ms
llama_perf_context_print: prompt eval time =      83.92 ms /    43 tokens (    1.95 ms per token,   512.37 tokens per second)
llama_perf_context_print:        eval time =    8299.35 ms /   219 runs   (   37.90 ms per token,    26.39 tokens per second)
llama_perf_context_print:       total time =    8597.25 ms /   262 tokens
llama_perf_context_print:    graphs reused =        217



Ответ Saiga:
Для настройки AIogram бота можно использовать следующую архитектуру файлов:

1. main.py - основный файл, где будет создан экземпляр класса Bot и в нем будет запущен метод polling() для начала работы бота.
2. config.py - файл с настройками бота, такими как токен, адрес базы данных, настройки прокси и т. д.
3. handlers.py - файл с обработчиками сообщений, где будут определены функции для различных команд и событий бота.
4. models.py - файл с моделями данных, которые используются в боте (например, для хранения информации об пользователях).
5. utils.py - фа

Генерация ответов завершена.


,source,question,reference_answer,model_answer
0,yandex_q,Мне нужно найти название фильма. В нем детей и...,Конечно. Навскидку я могу предложить несколько...,"Да, конечно, я могу помочь. Фильм с таким сюже..."
1,yandex_q,"Я слышал, что Петр I ввёл крепостное право в Р...","Да, Петр I действительно ввёл крепостное право...","Да, это правда. Петр I ввёл крепостное право в..."
2,ru_stackoverflow,"Я хотел бы узнать больше о том, как использова...","Конечно, с удовольствием помогу вам! Счётчики ...",Для проверки результата с помощью счётчиков мо...
3,yandex_q,Я хочу добавить 4 гб оперативной памяти на сво...,Для вашего ноутбука подходит оперативная памят...,Лучше всего подойдет модуль оперативной памяти...
4,ru_stackoverflow,"Я хочу настроить aiogram бота и не знаю, как п...","Конечно, я готов помочь вам с этим. Давайте на...",Для настройки AIogram бота можно использовать ...


## Оценка качества ответов модели

Для оценки качества я сравниваю эталонные ответы из датасета `ru_turbo_saiga` с ответами, которые сгенерировала модель Saiga.

Для сравнения используется модель эмбеддингов `paraphrase-multilingual-MiniLM-L12-v2`, которая переводит тексты в векторы.

Затем считается cosine similarity между:

- эталонным ответом;
- ответом модели Saiga.

Если similarity больше 0.5, я считаю ответ модели достаточно похожим на эталонный.

Важно: такая оценка не является идеальной, потому что модель Saiga могла уже видеть этот датасет при обучении. Кроме того, cosine similarity оценивает смысловую похожесть, но не всегда проверяет фактическую правильность ответа.

In [ ]:
# Оценка качества через cosine similarity

from sentence_transformers import SentenceTransformer, util
import pandas as pd

EMBEDDING_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

print("Загружаем модель для эмбеддингов:")
print(EMBEDDING_MODEL_NAME)

embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

similarities = []

for index, row in eval_results_df.iterrows():
    reference_answer = row["reference_answer"]
    model_answer = row["model_answer"]

    reference_embedding = embedding_model.encode(reference_answer, convert_to_tensor=True)
    model_embedding = embedding_model.encode(model_answer, convert_to_tensor=True)

    similarity = util.cos_sim(reference_embedding, model_embedding).item()
    similarities.append(similarity)

eval_scores_df = eval_results_df.copy()
eval_scores_df["similarity"] = similarities
eval_scores_df["is_similar"] = eval_scores_df["similarity"] > 0.5

accuracy = eval_scores_df["is_similar"].mean()

print("Результаты оценки:")
print(f"Количество примеров: {len(eval_scores_df)}")
print(f"Порог similarity: 0.5")
print(f"Accuracy: {accuracy:.2%}")

display(
    eval_scores_df[
        [
            "source",
            "question",
            "reference_answer",
            "model_answer",
            "similarity",
            "is_similar"
        ]
    ]
)

Загружаем модель для эмбеддингов:
sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Результаты оценки:
Количество примеров: 5
Порог similarity: 0.5
Accuracy: 80.00%


,source,question,reference_answer,model_answer,similarity,is_similar
0,yandex_q,Мне нужно найти название фильма. В нем детей и...,Конечно. Навскидку я могу предложить несколько...,"Да, конечно, я могу помочь. Фильм с таким сюже...",0.613261,True
1,yandex_q,"Я слышал, что Петр I ввёл крепостное право в Р...","Да, Петр I действительно ввёл крепостное право...","Да, это правда. Петр I ввёл крепостное право в...",0.721271,True
2,ru_stackoverflow,"Я хотел бы узнать больше о том, как использова...","Конечно, с удовольствием помогу вам! Счётчики ...",Для проверки результата с помощью счётчиков мо...,0.502386,True
3,yandex_q,Я хочу добавить 4 гб оперативной памяти на сво...,Для вашего ноутбука подходит оперативная памят...,Лучше всего подойдет модуль оперативной памяти...,0.817848,True
4,ru_stackoverflow,"Я хочу настроить aiogram бота и не знаю, как п...","Конечно, я готов помочь вам с этим. Давайте на...",Для настройки AIogram бота можно использовать ...,0.434340,False


## Итоговый вывод

В ходе домашней работы я получил практические навыки работы с большими языковыми моделями и платформой Hugging Face.

### Что было сделано

1. Я создал учетную запись на Hugging Face.
2. Получил Hugging Face token для доступа к моделям и датасетам.
3. Проверил доступ к репозиторию модели `IlyaGusev/saiga_mistral_7b_gguf`.
4. Загрузил квантованную модель Saiga Mistral 7B в формате GGUF.
5. Использовал файл модели `model-q4_K.gguf`.
6. Сгенерировал 3 шутки с помощью модели Saiga.
7. Загрузил датасет `IlyaGusev/ru_turbo_saiga`.
8. Подготовил 5 тестовых примеров из датасета.
9. Сгенерировал ответы Saiga на вопросы из датасета.
10. Сравнил ответы модели с эталонными ответами через cosine similarity.

### Результаты генерации шуток

Модель Saiga успешно сгенерировала 3 шутки на русском языке:

- шутка про программиста;
- шутка про нейросети и фриланс;
- шутка про Hugging Face и Google Colab.

Это показывает, что модель была загружена корректно и способна генерировать текст на русском языке.

### Результаты оценки качества

Для оценки качества я использовал 5 примеров из датасета `ru_turbo_saiga`.

Для каждого примера были взяты:

- вопрос пользователя;
- эталонный ответ из датасета;
- ответ, сгенерированный моделью Saiga.

Затем я сравнил эталонный ответ и ответ модели с помощью эмбеддингов `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`.

В качестве метрики использовалась cosine similarity.

Порог похожести был выбран:

```text
similarity > 0.5